# 🔬 Notebook 3: Typeahead — Deep Dive (bad → best + benchmarks)


## 🛠️ Setup

```bash
cd 06-system-designs/typeahead-autocomplete
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Three implementations, same API

We'll build three versions of `suggest(prefix) → top-K`, from naive to production-ish,
and benchmark them on the same corpus.

| Version | Idea | Query time | Build time | Memory |
|---|---|---|---|---|
| **1. Linear scan** | Loop every term | O(N·L + R log K) | O(N) | O(N) |
| **2. Sorted + bisect** | Sort terms, binary search the prefix range | O(L·log N + **R log K**) | O(N·L·log N) | O(N) |
| **3. Trie + precomputed top-K** | Tree of prefixes, top-K cached per node | **O(L)** | O(N·L) | O(N·L·**K**) |

- N = number of terms, L = prefix length, R = number of terms matching the prefix,
  K = how many suggestions we return.
- The string comparisons inside `bisect` each cost O(L), hence the `L·log N`.
- **The term that matters is `R`.** Binary search finds the matching range
  instantly, but you still have to *rank* everything in it. For a 1-character
  prefix, R ≈ N/26 — so version 2 collapses back toward linear exactly on the
  prefixes that get the most traffic. The benchmark below shows this happening.
- The trie's memory column carries a `K` because we cache a K-long list **at
  every node**. That factor is not a rounding error; we measure it later in
  this notebook and then fix it.

In [ ]:
# 🏭 Shared corpus + normalizer — reused in every version below.
import random, string, time, unicodedata, bisect
from collections import defaultdict
from heapq import nlargest

def normalize(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return " ".join(s.lower().strip().split())

def make_corpus(n=200_000, seed=7):
    """n distinct terms with n *distinct* popularity scores.

    Distinct scores matter: with ties, three implementations can each return a
    different-but-equally-valid top-K and we could not assert they agree."""
    random.seed(seed)
    words: set[str] = set()
    while len(words) < n:
        length = random.randint(3, 12)
        words.add("".join(random.choices(string.ascii_lowercase, k=length)))
    terms = sorted(words)
    scores = list(range(1, n + 1))
    random.shuffle(scores)
    return list(zip(terms, scores))

CORPUS = make_corpus()
print(f"corpus size: {len(CORPUS):_} terms")

## Version 1 — linear scan (bad)

In [ ]:
def suggest_linear(corpus, prefix, k=5):
    prefix = normalize(prefix)
    hits = [(t, s) for t, s in corpus if t.startswith(prefix)]
    return nlargest(k, hits, key=lambda x: x[1])

print(suggest_linear(CORPUS, "ab", k=3))


## Version 2 — sorted list + bisect (better)

If the term list is **sorted**, all terms starting with `prefix` form a contiguous
slice. We find that slice with two binary searches, then sort the slice by score.

This is exactly how a Redis `ZRANGEBYLEX` on a sorted set finds a prefix range.


In [ ]:
def prefix_upper_bound(prefix: str) -> str:
    """Smallest string that sorts strictly after every string starting with `prefix`.

    The tempting `prefix + "￿"` is WRONG: Python strings go up to U+10FFFF,
    so a term like "goo🍕" sorts *after* "goo￿" and silently disappears from
    the range — and "handle emoji" is one of our stated requirements. Bumping the
    last character by one is exact for every codepoint.
    """
    return prefix[:-1] + chr(ord(prefix[-1]) + 1)


class SortedIndex:
    def __init__(self, corpus):
        # sort by term ascending; store terms + parallel score list.
        pairs = sorted(corpus, key=lambda x: x[0])
        self.terms = [t for t, _ in pairs]
        self.scores = [s for _, s in pairs]

    def suggest(self, prefix: str, k=5):
        prefix = normalize(prefix)
        if not prefix:
            return []
        # all strings starting with prefix lie in [prefix, prefix_upper_bound(prefix))
        lo = bisect.bisect_left(self.terms, prefix)
        hi = bisect.bisect_left(self.terms, prefix_upper_bound(prefix))
        # then pick top-k by score in that window — this is the O(R) part
        window = zip(self.terms[lo:hi], self.scores[lo:hi])
        return nlargest(k, window, key=lambda x: x[1])

    def range_size(self, prefix: str) -> int:
        """R — how many terms the prefix actually matches. Used by the benchmark."""
        prefix = normalize(prefix)
        return (bisect.bisect_left(self.terms, prefix_upper_bound(prefix))
                - bisect.bisect_left(self.terms, prefix))

IDX = SortedIndex(CORPUS)
print(IDX.suggest("ab", k=3))

# The emoji case the naive "￿" bound gets wrong:
emoji_idx = SortedIndex([("goo", 1), ("goo\U0001F355", 9), ("gop", 5)])
assert emoji_idx.suggest("goo", k=5) == [("goo\U0001F355", 9), ("goo", 1)]
assert len([t for t in emoji_idx.terms if t < "goo￿"]) == 1   # the buggy bound loses one
print("✅ prefix range keeps astral-plane characters (emoji, CJK ext, …)")

## Version 3 — trie with precomputed top-K (best for query time)

A **trie** (prefix tree) shares common prefixes between terms:

```
   (root)
    /  \
   p    g
   |    |\
   y    o o
   |    | |
   th   ...
```

Each node caches its **top-K descendants**, computed once at build time.
At query time we walk `len(prefix)` edges and return the cached list.
That's the whole trick.


In [ ]:
class TrieNode:
    __slots__ = ("children", "is_end", "score", "top_k")
    def __init__(self):
        self.children: dict[str, "TrieNode"] = {}
        self.is_end = False
        self.score = 0
        self.top_k: list[tuple[str, int]] | None = None   # None = not cached here

class Trie:
    def __init__(self, k=5):
        self.root = TrieNode()
        self.k = k

    def add(self, term: str, freq: int = 1):
        term = normalize(term)
        node = self.root
        for ch in term:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True
        node.score += freq

    def precompute_top_k(self, max_cache_depth: int | None = None):
        """Post-order DFS: each node merges its own term (if any) with the top_k
        of its children and keeps the best K.

        `max_cache_depth` caps how deep we *store* the answer. Deeper nodes still
        return correct results — they just recompute on the fly (see `suggest`),
        which is cheap because a deep prefix has few descendants."""
        def dfs(node: TrieNode, prefix: str, depth: int):
            merged: list[tuple[str, int]] = []
            if node.is_end:
                merged.append((prefix, node.score))
            for ch, child in node.children.items():
                merged.extend(dfs(child, prefix + ch, depth + 1))
            best = nlargest(self.k, merged, key=lambda x: x[1])
            if max_cache_depth is None or depth <= max_cache_depth:
                node.top_k = best
            else:
                node.top_k = None
            return best
        dfs(self.root, "", 0)

    def _collect(self, node: TrieNode, prefix: str, out: list):
        """On-the-fly walk of a subtree — the fallback for uncached deep nodes."""
        if node.is_end:
            out.append((prefix, node.score))
        for ch, child in node.children.items():
            self._collect(child, prefix + ch, out)

    def suggest(self, prefix: str, k: int | None = None):
        prefix = normalize(prefix)
        node = self.root
        for ch in prefix:
            node = node.children.get(ch)
            if node is None:
                return []
        k = self.k if k is None else min(k, self.k)
        if node.top_k is not None:
            return node.top_k[:k]                 # O(1): the cached answer
        out: list[tuple[str, int]] = []
        self._collect(node, prefix, out)          # deep node → tiny subtree
        return nlargest(k, out, key=lambda x: x[1])

    def stats(self):
        """Count nodes and how many of them carry a cached top-K list."""
        nodes = cached = 0
        stack = [self.root]
        while stack:
            n = stack.pop()
            nodes += 1
            if n.top_k is not None:
                cached += 1
            stack.extend(n.children.values())
        return nodes, cached

trie = Trie(k=5)
for term, freq in CORPUS:
    trie.add(term, freq)
trie.precompute_top_k()
print(trie.suggest("ab"))

# All three implementations must agree, or the benchmark is meaningless.
for pfx in ["a", "ab", "qu", "xyz", "zz"]:
    assert trie.suggest(pfx) == IDX.suggest(pfx, k=5) == suggest_linear(CORPUS, pfx, k=5), pfx
print("✅ linear / sorted+bisect / trie return identical top-5 for every prefix tested")

## Benchmark all three

Same corpus, same prefixes, same K. The numbers will vary by machine, but the
**ratios** are what matter.


In [ ]:
def bench(fn, prefixes, repeats=5):
    t0 = time.perf_counter()
    for _ in range(repeats):
        for p in prefixes:
            fn(p)
    return (time.perf_counter() - t0) / (repeats * len(prefixes)) * 1_000_000  # µs

# Two prefix sets. This is the whole point of the experiment.
long_prefixes  = ["abc", "qui", "xyl", "hea", "thi", "neu", "gor"]   # 3 chars — rare
short_prefixes = ["a", "e", "s", "t"]                                # 1 char — the HOT ones

for name, pfxs in [("3-char", long_prefixes), ("1-char", short_prefixes)]:
    r = sum(IDX.range_size(x) for x in pfxs) / len(pfxs)
    print(f"{name} prefixes: average R (terms matching the prefix) = {r:,.0f}")
print()

rows = []
for name, pfxs in [("3-char prefixes", long_prefixes), ("1-char prefixes", short_prefixes)]:
    rows.append((
        name,
        bench(lambda p: suggest_linear(CORPUS, p, k=5), pfxs),
        bench(lambda p: IDX.suggest(p, k=5),            pfxs),
        bench(lambda p: trie.suggest(p),                pfxs),
    ))

print(f"{'':<18}{'linear scan':>14}{'sorted+bisect':>16}{'trie + top-K':>15}")
for name, a, b, c in rows:
    print(f"{name:<18}{a:>12,.0f} µs{b:>14,.0f} µs{c:>13,.1f} µs")
print()
for name, a, b, c in rows:
    print(f"{name}: trie is {a/c:>7,.0f}x faster than linear, {b/c:>6,.0f}x faster than sorted+bisect")

> 🎯 **This is the answer to "why precompute instead of ranking at query time?"**
>
> Look at the two rows. On rare 3-character prefixes, `sorted+bisect` is
> perfectly respectable — binary search lands on a handful of terms and ranking
> them is free. On 1-character prefixes it falls off a cliff, because R (the
> number of matching terms) is ~N/26 and you must score every one of them.
>
> And short prefixes are *exactly* the ones users type most: every single search
> passes through "a", then "ab", then "abc". Query-time ranking is fastest where
> traffic is thinnest and slowest where traffic is heaviest — precisely backwards.
>
> The trie is flat across both rows because it does **no work proportional to
> N or R** — only to the length of the prefix. All three implementations return
> identical answers (asserted above); they differ only in when the ranking work
> happens: once, offline, per prefix — or again and again, per keystroke.

## What the precomputed top-K actually costs

Nothing is free. We traded query time for **build time and memory**, and the
memory bill is bigger than it looks: we cache a K-element list at *every* node,
including the millions of deep nodes that will never be typed.

Let's measure the real trie we just built rather than guess.

In [ ]:
NODE_BYTES = 24            # a compact (C++/Rust) node: char, child pointer(s), flags
TOPK_BYTES = 5 * 8         # K x (4-byte term id + 4-byte score)

nodes, cached = trie.stats()
print(f"terms            : {len(CORPUS):,}")
print(f"trie nodes       : {nodes:,}   ({nodes/len(CORPUS):.1f} nodes per term)")
print(f"nodes with top-K : {cached:,}  ({cached/nodes:.0%})")
print(f"structure        : {nodes*NODE_BYTES/1e6:,.0f} MB")
print(f"top-K caches     : {cached*TOPK_BYTES/1e6:,.0f} MB   <- larger than the trie itself")
print()

# Snapshot the answers so we can prove the optimisation changes nothing.
probe = ["a", "ab", "abc", "qu", "xyl", "zzz", "th"]
before = {x: trie.suggest(x) for x in probe}

# --- The fix: only cache where the traffic is. -----------------------------
# A prefix of length <= 4 is typed constantly and has a huge subtree, so caching
# pays. A prefix of length 8 is typed rarely and has almost no descendants, so
# walking its subtree on demand is cheap.
CACHE_DEPTH = 4
t0 = time.perf_counter()
trie.precompute_top_k(max_cache_depth=CACHE_DEPTH)
build_s = time.perf_counter() - t0

nodes2, cached2 = trie.stats()
after = {x: trie.suggest(x) for x in probe}
assert after == before, "depth-capped cache must not change a single answer"

print(f"with top-K cached only to depth {CACHE_DEPTH}:")
print(f"nodes with top-K : {cached2:,}  ({cached2/nodes2:.1%} of nodes)")
print(f"top-K caches     : {cached2*TOPK_BYTES/1e6:,.1f} MB "
      f"(was {cached*TOPK_BYTES/1e6:,.0f} MB — {1 - cached2/cached:.1%} smaller)")
print(f"rebuild took     : {build_s:.2f}s")
print("✅ identical suggestions for every probed prefix")
print()

t_short = bench(lambda p: trie.suggest(p), ["a", "e", "s", "t"])
t_deep  = bench(lambda p: trie.suggest(p), ["abcde", "qwert", "zzzzz"])
print(f"query, cached depth<=4 prefixes : {t_short:8,.1f} µs   (still O(1) lookup)")
print(f"query, uncached deep prefixes   : {t_deep:8,.1f} µs   (on-the-fly subtree walk)")

**The trade-off, stated honestly:** the depth cap makes deep prefixes slower —
they now walk a subtree instead of reading a cached list. That is acceptable
only because subtree size shrinks fast with depth, and because those prefixes
are rare. If your corpus has a pathological branch (say every term starts with
`"amazon "`), the cap must follow the *shape of the data*, not a fixed number:
cache a node's top-K when its subtree is larger than some threshold.

Restore the full cache before moving on if you want the original timings back —
`trie.precompute_top_k()` with no argument.

## Updating the trie safely

**Never** mutate a live trie under concurrent reads — locking it per update is
how you miss your latency SLO.

Instead:
1. The live trie is treated as **immutable** once built.
2. A batch job reads recent query logs and builds a **new** trie off-box.
3. The service loads the new trie and does an **atomic pointer swap**.
4. The old trie is freed once all in-flight requests finish.

This is the exact same pattern as zero-downtime DB migrations or blue/green deploys.


## Ranking beyond raw popularity

Popularity alone goes stale. Real systems blend:

- **Frequency** over the last N days (baseline).
- **Trending**: rate of change, `d(frequency)/dt`.
- **Personalization**: user history, region, language — usually an overlay
  (a small per-user trie merged with the global top-K at query time).
- **Spell correction**: if prefix has no hits, try edit-distance-1 variants.

### Freshness via decaying counter

Instead of updating trie nodes every keystroke, we decay **counts in the
aggregator** on each rebuild cycle. The decay lives in the *pipeline that
feeds the next snapshot*, not the serving trie.


In [ ]:
# 📉 Tiny exponential decay example (this lives in the aggregator, not the serving trie).
class DecayingCounter:
    def __init__(self, half_life_s=3600):
        self.half_life = half_life_s
        self.count = 0.0
        self.ts = time.time()

    def add(self, n=1):
        self._decay()
        self.count += n

    def value(self):
        self._decay()
        return self.count

    def _decay(self):
        now = time.time()
        dt = now - self.ts
        if dt > 0:
            self.count *= 0.5 ** (dt / self.half_life)
            self.ts = now

c = DecayingCounter(half_life_s=0.5)
c.add(100); time.sleep(0.25); print("after 0.25s:", round(c.value(), 2))
time.sleep(0.5);             print("after 0.75s:", round(c.value(), 2))


## Optional appendix — typo tolerance

If the user types a prefix with no matches, fall back to candidates at
**edit distance 1** from the prefix (one insertion / deletion / substitution).
Real systems use richer structures (BK-tree, Symspell), but for a prefix
autocomplete an edit-distance-1 generator gets you 80% of the way.


In [ ]:
def edits1(s, alphabet="abcdefghijklmnopqrstuvwxyz "):
    splits = [(s[:i], s[i:]) for i in range(len(s) + 1)]
    deletes    = [a + b[1:]         for a, b in splits if b]
    substitutes= [a + c + b[1:]     for a, b in splits if b for c in alphabet]
    inserts    = [a + c + b         for a, b in splits      for c in alphabet]
    return set(deletes + substitutes + inserts)

# Use a small curated trie so the demo clearly shows the fuzzy fallback in action.
fuzzy_trie = Trie(k=5)
for term, freq in [
    ("python", 900), ("python tutorial", 500),
    ("google", 800), ("google docs", 400),
    ("javascript", 700), ("java", 600),
]:
    fuzzy_trie.add(term, freq)
fuzzy_trie.precompute_top_k()

def suggest_with_fuzzy(t: Trie, prefix: str, k=5):
    hits = t.suggest(prefix, k=k)
    if hits:
        return hits
    merged: list[tuple[str, int]] = []
    for candidate in edits1(normalize(prefix)):
        merged.extend(t.suggest(candidate, k=k))
    merged = list(dict.fromkeys(merged))     # the same term can come from several edits
    return nlargest(k, merged, key=lambda x: x[1])

# 'gogle' is 'google' with one 'o' missing — a real edit-distance-1 typo.
print("direct 'gogle'  →", fuzzy_trie.suggest("gogle"))
print("fuzzy  'gogle'  →", suggest_with_fuzzy(fuzzy_trie, "gogle"))
print("fuzzy  'pythoon'→", suggest_with_fuzzy(fuzzy_trie, "pythoon"))
print("fuzzy  'javscr' →", suggest_with_fuzzy(fuzzy_trie, "javscr"))

## Memory & sharding — with the numbers from the measurement above

Extrapolate what we just measured. Our 200 k-term trie (avg 7.5 chars) used
**~4.6 nodes per term**: the top `log₂₆(200 000) ≈ 3.7` levels are saturated and
shared by everyone, so each term only contributes its own tail. Run the same
model on a 100 M-term English corpus at 12 chars average — the estimate in
Notebook 1 — and you get `100 M × (12 − log₂₆(100 M)) ≈ **630 M nodes**`,
about 6.3 per term.

| | bytes/node | total at 630 M nodes |
|---|---|---|
| Node structure (compact C++) | 24 | ~15 GB |
| Top-K cached at **every** node (K=5) | 40 | ~26 GB |
| **Total** | 64 | **~41 GB** |

So: **not** "a few GB", and **not** one box. Three consequences fall out of that
one number:

- **Shard by first 1–2 characters.** 26 shards of ~1.6 GB each is comfortable;
  the suggest service routes by prefix. The cost: a prefix shorter than the
  shard key (the empty query) has to fan out to every shard and merge.
- **Cap the top-K cache by depth** (measured above) — it is the majority of the
  memory and most of it is never read. This alone roughly halves the footprint.
- **One trie per language.** Keeps each shard small *and* stops Turkish
  dotless-i normalisation rules from corrupting English suggestions.

**Replication vs sharding, honestly:** sharding is forced on you by memory,
replication by QPS — they are different problems and you will need both.
Each shard needs enough replicas to serve its share of peak QPS, so total boxes
= shards × replicas, and the trie snapshot has to be shipped to every one of
them on each rebuild. That snapshot distribution (tens of GB, every 5–15 min)
is a real operational cost that the "just keep it in memory" pitch never mentions.

For personalization, a small per-user trie is merged into the global top-K at
query time (union + `nlargest`) — cheap, because both inputs are already K-sized.